In [ ]:
## a useful function to generate a data list for further analysis
import os,json,sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from analyse_stimulus_evoked_response import main
##need to add this additional cell because useful tools are in another folder. Need to integrate these two folders one day
current_working_directory = Path.cwd()
parent_dir = current_working_directory.resolve().parents[0]
sys.path.insert(0, str(parent_dir) + "\\utilities")
from useful_tools import select_animals_gpt, find_file
from data_cleaning import preprocess_fictrac_data

Session 0.1: Load analysis methods in python dictionary form

In [ ]:
#sheet_name="Zball"
sheet_name="MatrexVR"
json_file = "./analysis_methods_dictionary.json"
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())
    

Datasets="Z:/DATA/experiment_trackball_Optomotor"
this_dataset_dir = f"{Datasets}/{sheet_name}"

In [ ]:
analysis_methods

In [ ]:
analysis_methods.update({"experiment_name": "long_densities"})
analysis_methods.update({"stim_duration": 60})
analysis_methods.update({"interval_duration": 60})

Session 0.2: Load animal directory as a list

In [ ]:
## this cell searches for a folder with a specified experiment_name under the dataset path and list up all the dat file in that folder.
## In this project, we usually have one dat file in that folder so there is no confusion
dir_list = []
file_type=".dat"
for root, dirs, files in os.walk(this_dataset_dir):
    if analysis_methods.get("experiment_name") in root.split(
        os.path.sep
    ):  ## add this condition to avoid data from other experiments
        for folder in dirs:
            if folder.startswith("session"):
                folder_path=os.path.join(root,folder)
                if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                    dir_list.append(folder_path.replace("\\", "/"))
print(f"these directories are found {dir_list}")

Session 1.0: Create curated dataset based on the list of directories

In [ ]:
pattern="database*.pickle"
for this_dir in dir_list[24:]:
    if any(Path(this_dir).glob(pattern)):
        print(f"curated fictrac data found in {this_dir}. Skip this file")
        continue
    else:
        print(f"no curated fictrac data in {this_dir}. Create curated file")
        preprocess_fictrac_data(this_dir,analysis_methods)

In [ ]:
# Define the path to your Excel file
dir_list = []
# choose folders that contains fictrac .dat file
file_type=".dat"
using_google_sheet=True
sheet_name = "MatrexVR"
experiment_name=analysis_methods.get("experiment_name")
if analysis_methods.get("load_experiment_condition_from_database") == True:
    if using_google_sheet==True:
        database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8"
                #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8/edit?usp=sharing
        url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        #df = pd.read_excel(url, engine='openpyxl')## use this function if the file is not google sheet but uploaded excel file

        df = pd.read_csv(url)
    else:
        excel_file_path = "Z:/DATA/experiment_trackball_Optomotor/Locusts Management.xlsx"
        print(f"using a database {excel_file_path} from the server but this file might be outdated")
        # Create a 'with' statement to open and read the Excel file
        with pd.ExcelFile(excel_file_path) as xls:
            # Read the Excel sheet into a DataFrame with the sheet name (folder name)
            df = pd.read_excel(xls, sheet_name)
        ##list up the conditions and answers as strings for input argument to select animal. One condition must pair with one answer
    if analysis_methods.get("select_animals_by_condition") == True:
        animal_of_interest=select_animals_gpt(df,"Experimenter","CL","Exluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #print(animal_of_interest)
    else:
        animal_of_interest=df
    ID_array=animal_of_interest["ID"].values
    dir_list = [
    root.replace("\\", "/")
    for root, dirs, files in os.walk(this_dataset_dir)
    if any(ID in root for ID in ID_array)
    and experiment_name in root.split(os.path.sep)
    and any(name.endswith(file_type) for name in files)
]
else:
    ## this cell searches for a folder with a specified experiment_name under the dataset path and list up all the hdf5 file in that folder.
    ## However,some changes need to be made once we do sleap or deeplabcut where there are more than one H5 file generated
    for root, dirs, files in os.walk(this_dataset_dir):
        if analysis_methods.get("experiment_name") in root.split(os.path.sep):## add this condition to avoid data from other experiments
            if any(name.endswith(file_type) for name in files):
                dir_list.append(root.replace("\\", "/"))

In [ ]:
dir_list[24:]

In [ ]:
new_list=dir_list[19:-9] ##for density experiment in gregarious conditions

In [ ]:
len(new_list)

In [ ]:
new_list=dir_list[1:4]+dir_list[5:9]+dir_list[14:16]+dir_list[20:]

In [ ]:
new_list=dir_list[:6]+dir_list[7:-9]

In [ ]:
LS_list=dir_list[-9:]

In [ ]:
LS_list

In [ ]:
new_list=dir_list+LS_list

In [ ]:
new_list=new_list[:7]+new_list[8:]


In [ ]:
new_list

In [ ]:
len(new_list)

Session 2.1: Analyse optomotor response

In [ ]:
# because the import does not update the new version of python.
# Need to restart kernel 
output0_across_exp=[]
output1_across_exp=[]
output2_across_exp=[]
output3_across_exp=[]
output4_across_exp=[]
for this_dir in dir_list[:8]:
    if "archive" in this_dir:
        print(f"skip archive folder for {this_dir}")
        continue
    summary,speed,rotation,travel_distance_whole_session=main(this_dir,analysis_methods)
    output0_across_exp.append(summary)
    output1_across_exp.append(speed)
    output2_across_exp.append(rotation)
    output3_across_exp.append(travel_distance_whole_session)
    output4_across_exp.append(this_dir)

In [ ]:
df2=pd.DataFrame(pd.concat(output0_across_exp))

In [ ]:
df2

In [ ]:
df

In [ ]:
#output0_across_exp[13]
growth_condition='gregarious'
df2['file_path'] = np.repeat(output4_across_exp,output0_across_exp[0].shape[0])
df2['growth_condition'] = np.repeat(growth_condition,output0_across_exp[0].shape[0]*len(output0_across_exp))
df2['ID'] = np.repeat(np.arange(len(output4_across_exp)),output0_across_exp[0].shape[0])


In [ ]:
frames=[df2,df]
result = pd.concat(frames)

In [ ]:
result

In [ ]:
file_name=f"zball_database.csv"
df.to_csv(file_name)

Session 2.2: Analyse optomotor response with multi-engines

In [ ]:
##this cell start the multi-engines. Make sure to run only once
import time
import ipyparallel as ipp
def show_clusters():
    clusters = ipp.ClusterManager().load_clusters() 
    print("{:15} {:^10} {}".format("cluster_id", "state", "cluster_file")) 
    for c in clusters:
        cd = clusters[c].to_dict()
        cluster_id = cd['cluster']['cluster_id']
        controller_state = cd['controller']['state']['state']
        cluster_file = getattr(clusters[c], '_trait_values')['cluster_file']
        print("{:15} {:^10} {}".format(cluster_id, controller_state, cluster_file))
    return cluster_id

cluster = ipp.Cluster(n=6)
await cluster.start_cluster()
cluster_neuropc=show_clusters()

In [ ]:
##input cluster_id from previous cell
rc = ipp.Client(cluster_id=cluster_neuropc)

# Create a DirectView for parallel execution
dview = rc.direct_view()

# Define a function for parallel processing
def process_directory(thisDir, analysis_methods):
    from analyse_stimulus_evoked_response import main
    # Check if the H5 file (curated dataset) exists
    summary,speed,rotation = main(thisDir, analysis_methods)
    return (summary,speed,rotation)

# Define analysis_methods

# Use parallel execution to process directories
results = dview.map_sync(process_directory, dir_list, [analysis_methods] * len(dir_list))

# Initialize result lists
output0_across_exp=[]
output1_across_exp=[]
output2_across_exp=[]

# Collect and organize results
for result in results:
    if result is not None:
        summary,speed,rotation = result
        output0_across_exp.append(summary)
        output1_across_exp.append(speed)
        output2_across_exp.append(rotation)

# Now, you have the results collected in the respective lists

In [ ]:
rc.shutdown()

Session 2.3: Load existing dataset

In [ ]:
database_pattern = "*database.csv"
load_existing_dataset=True
this_database = find_file(this_dataset_dir, database_pattern)
individual_readout_pattern="*velocity.npy"
print(this_database)
if this_database is None or this_database.is_file()==False:
    print("no post-analysis database created. Either run analysis cell to get the list of animal's response or assemble post-analysis dataset from individual animals")
    if 'output0_across_exp' in locals():
        df=pd.DataFrame(pd.concat(output0_across_exp))
        print("use local list created from jupyter notebook")
    else:
        for this_dir in dir_list[:3]:
            this_file = find_file(this_dir, individual_readout_pattern)
            if this_file.is_file()==True:
                print({f"npy file is found. Let's assemble them"})
                arr=np.load(this_file)
                print(arr.shape)
            else:
                print({f"no npy file created in this path: {this_dir}. Skip this animal"})
                continue
elif load_existing_dataset==True:
    print("load csv into pandas dataframe")
else:
    print("do not know what to do here")

In [ ]:
df

In [ ]:
growth_condition='gregarious'
df['file_path'] = np.repeat(output4_across_exp,output0_across_exp[0].shape[0])
df['growth_condition'] = np.repeat(growth_condition,output0_across_exp[0].shape[0]*len(output0_across_exp))
df['ID'] = np.repeat(np.arange(len(output4_across_exp)),output0_across_exp[0].shape[0])

In [ ]:
path = 'orientation_database.csv'
df = pd.read_csv(path,index_col=0)

In [ ]:
mean_travel_distance_trials=df.groupby(['ID','Orientation'])['travel_distance'].mean().values

In [ ]:
mean_travel_distance_trials

In [ ]:
mean_travel_distance_trials[::2]
np.savetxt("inward.csv", mean_travel_distance_trials[::2], delimiter=",")

In [ ]:
mean_travel_distance_trials[1::2]
np.savetxt("outward.csv", mean_travel_distance_trials[1::2], delimiter=",")

In [ ]:
np.divide(mean_travel_distance_trials[::2],mean_travel_distance_trials[1::2])

In [ ]:
plt.scatter(mean_travel_distance_trials[::2],mean_travel_distance_trials[1::2])
plt.xlim([0, 1500])
plt.ylim([0, 1500])
ax = plt.gca()
ax.set_aspect('equal', adjustable='box')
plt.draw()

In [ ]:
num_bins = 350
n, bins, patches = plt.hist(df['travel_distance'],num_bins)
bins

In [ ]:
dfsel = df[df['travel_distance']>30]
file_name=f"orientation_database_stationary30.csv"
dfsel.to_csv(file_name)

In [ ]:
dfsel_balance_travel_distance=dfsel[dfsel['ID'].isin([3,4,8,9,10,11,12,13,14])]
file_name=f"orientation_database_stationary30_balance_travel_distance.csv"
dfsel_balance_travel_distance.to_csv(file_name)

In [ ]:
df[df['travel_distance']>150]

In [ ]:
for this_group in df["growth_condition"].unique():
    d_con=df[(df["growth_condition"]==this_group) & (df["travel_distance"]<200)]
    fig, axes = plt.subplots(
        nrows=1, ncols=len(d_con["stim_type"].unique()), figsize=(25, 3), sharey=True,tight_layout=True
    )
    for this_stim, ax in zip(np.sort(d_con["stim_type"].unique()),axes.reshape(-1)):
        ax.scatter(d_con[(d_con["stim_type"]==this_stim) & (d_con["travel_distance"]>5)]["travel_distance"],d_con[(d_con["stim_type"]==this_stim)&(d_con["travel_distance"]>5)]["opto_index"])
        ax.set_xticks([0, 200])
        ax.set_yticks([-1, 1])
        ax.set_xlim([0, 200])
        ax.set_ylim([-1, 1])
        ax.set_box_aspect(1)
        ax.set_ylabel("Optomotor index")
        ax.set_xlabel("Travel distance (mm)")
        ax.set_title(f"Number of dots:{this_stim}")
    fig.suptitle(f"Growth condition: {this_group}", fontsize=16)
    fig_name=f"scatter_plot_{this_group}.png"
    fig.savefig(fig_name)

In [ ]:
df[df['travel_distance']>150]['file_path'].values

In [ ]:
dfsel = df[df['travel_distance']>5]
#df.hist('Median speed')

In [ ]:
file_name=f"density_database_no_stationary.csv"
dfsel.to_csv(file_name)

In [ ]:
plt.hist(output3_across_exp,bins=range(0,10000,1000))

In [ ]:
init_frame=analysis_methods.get("interval_duration")*analysis_methods.get("frame_rate")
culz_stim_onset=test4[:,init_frame]
DF=pd.DataFrame(culz_stim_onset)
file_name=f"culz_onset4.csv"
DF.to_csv(file_name)

In [ ]:
init_frame=analysis_methods.get("interval_duration")*analysis_methods.get("frame_rate")
prestim_analyse=5
prestim_5_sec=init_frame-prestim_analyse*analysis_methods.get("frame_rate")

In [ ]:
culz_stim_onset=output2_arr[:,init_frame]
culz_stim_onset_5sec_before=output2_arr[:,init_frame-prestim_5_sec]
this_response=culz_stim_onset-culz_stim_onset_5sec_before
DF=pd.DataFrame(this_response)
file_name=f"culz_onset.csv"
DF.to_csv(file_name)

Session 3: plotting data

In [ ]:

visual_paradigm_name= analysis_methods.get("experiment_name")
colormap = np.array(analysis_methods.get("graph_colour_code"))
fig2, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(18, 14), tight_layout=True
)
ax1, ax2, ax3, ax4 = axes.flatten()
for i in range(len(output0_across_exp)):
    this_animal = output0_across_exp[i]
    tmp=this_animal.groupby("stim_type").count()
    follow_count_coherence = tmp.index.values
    for j in range(len(this_animal.groupby("stim_type"))):
        this_coherence=follow_count_coherence[j]
        this_response = this_animal.loc[
            this_animal["stim_type"] == this_coherence, "opto_index"
        ].values
        # this_coherence = x_axis_value_running_trials[i]
        mean_response = np.mean(this_response, axis=0)
        sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(
            this_response.shape[0]
        )
        ax1.errorbar(
            this_coherence,
            mean_response,
            yerr=sem_response,
            c=colormap[5],
            fmt="o",
            elinewidth=2,
            capsize=3,
        )
    ax1.set_ylim(-1, 1)
    ax1.set(
        yticks=[-1, 0, 1],
        ylabel="Optomotor Index",
        xlabel=visual_paradigm_name,)
df.groupby(['ID'])['travel_distance'].sum().values

# for idx, above_threshold in enumerate(df.groupby(['ID'])['travel_distance'].sum()>threshold):
#     if above_threshold == True:
#         this_animal = df.loc[df['ID']==idx]
#         tmp=this_animal.groupby("stim_type").count()
#         follow_count_coherence = tmp.index.values
#         for j in range(len(this_animal.groupby("stim_type"))):
#             this_coherence=follow_count_coherence[j]
#             this_response = this_animal.loc[
#                 this_animal["stim_type"] == this_coherence, "opto_index"
#             ].values
#             # this_coherence = x_axis_value_running_trials[i]
#             mean_response = np.mean(this_response, axis=0)
#             sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(
#                 this_response.shape[0]
#             )
#             ax4.errorbar(
#                 this_coherence,
#                 mean_response,
#                 yerr=sem_response,
#                 c=colormap[5],
#                 fmt="o",
#                 elinewidth=2,
#                 capsize=3,
#             )
#         ax4.set_ylim(-1, 1)
#         ax4.set(
#             yticks=[-1, 0, 1],
#             ylabel="Optomotor Index",
#             xlabel=visual_paradigm_name,)
        #print(df.loc[df['ID']==idx])
    
    # ax4.scatter(follow_count_coherence, follow_count, c=colormap[0], marker="o")
    # ax4.set_ylim(0, 15)
    # ax4.set(
    #     yticks=[0, 15],
    #     ylabel="Follow response (count)",
    #     xticks=[100, 50, 0, -50, -100],
    #     xlabel="Coherence level (%)",
    # )
    ##following one dot (dot lifetime)
    ##memory part (30s)
    ##interval: rondot
    ##continous

In [ ]:
visual_paradigm_name= analysis_methods.get("experiment_name")
colormap = np.array(analysis_methods.get("graph_colour_code"))
#print(colormap)
fig2, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(18, 14), tight_layout=True
)
ax1, ax2, ax3, ax4 = axes.flatten()
df.groupby(['ID'])['travel_distance'].sum().values
num_bins=10
n, bins, patches = ax1.hist(df.groupby(['ID'])['travel_distance'].sum().values,num_bins,color=colormap[3])
ax1.set(
ylabel="Count",
xlabel="Walk distance (mm)")
threshold=10000
for idx, above_threshold in enumerate(df.groupby(['ID'])['travel_distance'].sum()>threshold):
    this_animal = df.loc[df['ID']==idx]
    tmp=this_animal.groupby("stim_type").count()
    follow_count_coherence = tmp.index.values
    for j in range(len(this_animal.groupby("stim_type"))):
        this_coherence=follow_count_coherence[j]
        this_response = this_animal.loc[
            this_animal["stim_type"] == this_coherence, "opto_index"
        ].values
        mean_response = np.mean(this_response, axis=0)
        sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(
            this_response.shape[0]
        )
        if above_threshold == True:
            this_colour=colormap[5]
        else:
            this_colour=colormap[6]
        ax2.errorbar(
            this_coherence,
            mean_response,
            yerr=sem_response,
            c=this_colour,
            fmt="o",
            elinewidth=2,
            capsize=3,
        )
ax2.set_ylim(-1, 1)

if visual_paradigm_name =="gratings":
    xlabel="Temporal frequency (Hz)"
elif visual_paradigm_name =="coherence":
    xlabel="Coherence (%)"
ax2.set(
    yticks=[-1, 0, 1],
    ylabel="Optomotor Index",
    xlabel=xlabel)
    
num_bins=100
n, bins, patches = ax3.hist(df['travel_distance'].values,num_bins,density=True,range=(0,50),color=colormap[3])
ax3.set(
ylabel="Density",
xlabel="Walk distance (mm)")
walk_threshold=20
df25=df[df['travel_distance']>walk_threshold]
for this_coherence in df25["stim_type"].unique():
    this_response = df25.loc[
    df25["stim_type"] == this_coherence, "opto_index"].values
    mean_response = np.mean(this_response, axis=0)
    sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(this_response.shape[0])
    ax4.errorbar(
        this_coherence,
        mean_response,
        yerr=sem_response,
        c=colormap[0],
        fmt="o",
        elinewidth=2,
        capsize=3,
    )
ax4.set_ylim(-1, 1)
ax4.set(
    yticks=[-1, 0, 1],
    ylabel="Optomotor Index",
    xlabel=xlabel)
fig2.suptitle(f'The response of locusts during {visual_paradigm_name} stimulus presentation')
ax1.title.set_text('The total walk distance of every locust')
ax2.title.set_text('The optomotor index of every locust')
ax3.title.set_text('The walk distance of every trial')
ax4.title.set_text('The optomotor index of every walk trial')